# Switching Cost Calibration Model

This notebook builds a **systematic switching-cost model** for your RL reward.

Given:
- spike power in W (you called this spike energy in W)
- startup delay in seconds

it outputs an **ideal cost** scaled to your current reward magnitude (from an evaluation CSV).

Model used:

$$C_{switch}=\lambda_P\,P_{spike}+\lambda_D\,T_{delay}$$

where $\lambda_P,\lambda_D$ are auto-calibrated from your current policy reward/SEC scale and a reference switch event.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

# Path to your evaluation CSV (edit if needed)
EVAL_CSV = Path('../experiments/results/A1_policy/lstm/seed_456/eval/evaluation_results_MlpLstmPolicy.csv')

assert EVAL_CSV.exists(), f'CSV not found: {EVAL_CSV}'
df = pd.read_csv(EVAL_CSV)

summary = {
    'rows': len(df),
    'sec_mean': float(df['sec'].mean()),
    'sec_median': float(df['sec'].median()),
    'sec_p90': float(df['sec'].quantile(0.90)),
    'reward_mean': float(df['reward'].mean()),
    'reward_median_abs': float(df['reward'].abs().median()),
    'reward_p90_abs': float(df['reward'].abs().quantile(0.90)),
    'switch_rate': float((df['switch'] == 1).mean()),
}

pd.Series(summary)

rows                 721.000000
sec_mean               0.007513
sec_median             0.007977
sec_p90                0.010614
reward_mean           -0.761114
reward_median_abs      0.799458
reward_p90_abs         1.076252
switch_rate            0.019417
dtype: float64

## Calibration Logic

We choose a reference target cost for a known switch event, then solve for coefficients.

1. Compute a target reference cost from your reward scale:
   - `cost_from_sec = sec_median * sec_equiv_steps`
   - `cost_cap = reward_median_abs * cap_fraction_of_reward`
   - `reference_cost = min(cost_from_sec, cost_cap)`

2. Split that cost between power and delay by `power_share` in [0,1].

3. Solve:

$$\lambda_P=\frac{power\_share\cdot C_{ref}}{P_{ref}},\quad \lambda_D=\frac{(1-power\_share)\cdot C_{ref}}{T_{ref}}$$

Then for any new measured event:

$$C_{ideal}=\lambda_P P_{new}+\lambda_D T_{new}$$

In [5]:
def calibrate_switch_model(
    sec_median: float,
    reward_median_abs: float,
    ref_spike_power_w: float,
    ref_delay_s: float,
    sec_equiv_steps: float = 15.0,
    cap_fraction_of_reward: float = 0.25,
    power_share: float = 0.80,
):
    """
    Returns a calibrated model dict with lambda_P, lambda_D and reference_cost.

    sec_equiv_steps: how many typical SEC-only steps a single switch should roughly equal.
    cap_fraction_of_reward: prevent switch cost from dominating typical per-step reward magnitude.
    power_share: fraction of the switch cost attributed to spike power term.
    """
    eps = 1e-12
    cost_from_sec = float(sec_median) * float(sec_equiv_steps)
    cost_cap = float(reward_median_abs) * float(cap_fraction_of_reward)
    reference_cost = min(cost_from_sec, cost_cap)

    power_share = float(np.clip(power_share, 0.0, 1.0))
    lambda_p = (power_share * reference_cost) / max(float(ref_spike_power_w), eps)
    lambda_d = ((1.0 - power_share) * reference_cost) / max(float(ref_delay_s), eps)

    return {
        'lambda_p': lambda_p,
        'lambda_d': lambda_d,
        'reference_cost': reference_cost,
        'cost_from_sec': cost_from_sec,
        'cost_cap': cost_cap,
    }

def ideal_switch_cost(spike_power_w: float, delay_s: float, model: dict) -> float:
    return model['lambda_p'] * float(spike_power_w) + model['lambda_d'] * float(delay_s)

In [9]:
# --- Reference event from your measurement report (OAI -> DPDK example) ---
REF_SPIKE_POWER_W = 34.81
REF_DELAY_S = 23.960448

model = calibrate_switch_model(
    sec_median=summary['sec_median'],
    reward_median_abs=summary['reward_median_abs'],
    ref_spike_power_w=REF_SPIKE_POWER_W,
    ref_delay_s=REF_DELAY_S,
    sec_equiv_steps=15.0,      # tune if needed
    cap_fraction_of_reward=0.25,
    power_share=0.80,
)

model

{'lambda_p': 0.002750036373832899,
 'lambda_d': 0.0009988207041571509,
 'reference_cost': 0.11966095771640402,
 'cost_from_sec': 0.11966095771640402,
 'cost_cap': 0.19986455142498016}

In [10]:
# === USER INPUT CELL ===
# Put your measured values here:
spike_power_w = 1.2   # W
delay_s = 23.960448     # s

cost = ideal_switch_cost(spike_power_w, delay_s, model)

print(f'Input spike_power_w: {spike_power_w:.6f} W')
print(f'Input delay_s:      {delay_s:.6f} s')
print(f'Ideal switch cost:  {cost:.6f}')

# Optional derived quantity (not used by model directly)
energy_j = spike_power_w * delay_s
print(f'Implied spike energy: {energy_j:.3f} J')

Input spike_power_w: 1.200000 W
Input delay_s:      23.960448 s
Ideal switch cost:  0.027232
Implied spike energy: 28.753 J


In [11]:
# Optional: quick sensitivity table
test_cases = pd.DataFrame({
    'spike_power_w': [20, 30, 40, 50],
    'delay_s': [10, 20, 30, 40],
})
test_cases['ideal_cost'] = test_cases.apply(
    lambda r: ideal_switch_cost(r['spike_power_w'], r['delay_s'], model), axis=1
)
test_cases

,spike_power_w,delay_s,ideal_cost
0,20,10,0.064989
1,30,20,0.102478
2,40,30,0.139966
3,50,40,0.177455


## How to Use This in `environment.py`

Use the resulting `ideal_cost` as your directional type-switch penalty (for example `OAI -> DPDK`).

If you want direction-aware costs, calibrate one model per direction (e.g., OAI->DPDK and DPDK->OAI) with each direction's measured reference event.